# Descargar librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ventanas_mejoras import expand_to_windows, build_extra_features, category_burst_report

# Leer y separar archivo

In [2]:
df = pd.read_csv('../data/logs.csv')
df = df.drop(columns=['_ignored'])
df_llm = df[df['_index'].str.startswith('llm-logs', na=False)]
df_llm = df_llm.drop(columns=['http_status_code', 'service_id', 'client_ip'])
df_system = df[df['_index'].str.startswith('sap-logs', na=False)]
df_system = df_system.drop(columns=['llm_error_message', 'llm_response_size_bytes', 'sap_llm_response_size', 'sap_llm_response_time', 'llm_prompt_tokens', 'llm_completion_tokens', 'llm_total_tokens', 'llm_response_time_ms', 'llm_model_id', 'llm_status', 'llm_cost_usd', 'llm_prompt_category', 'llm_temperature', 'llm_top_p', 'llm_stream', 'llm_finish_reason', 'llm_prompt_id', 'llm_provider', 'llm_prompt'])

In [3]:
hit = df['sap_function_message'].str.contains('inject|jailbreak|prompt', case=False, na=False)
print(df[hit].groupby('_index')['sap_function_message'].value_counts().head(20))

_index            sap_function_message                                                                                       
llm-logs-2026.04  LLM ERROR: Prompt length exceeds model context window. (model=o1-mini, app=sap-concur)                         10
                  LLM ERROR: Prompt injection detected and blocked. (model=gemini-1.5-pro, app=sap-s4hana-finance)                9
                  LLM ERROR: Prompt length exceeds model context window. (model=gpt-4, app=sap-data-sphere)                       9
                  LLM ERROR: System prompt exceeds maximum length. (model=o1, app=sap-analytics-cloud)                            8
                  LLM ERROR: Injection attempt identified and rejected. (model=claude-3-5-sonnet, app=sap-fiori-launchpad)        8
                  LLM ERROR: System prompt exceeds maximum length. (model=claude-3-7-sonnet, app=sap-fiori-launchpad)             8
                  LLM ERROR: Injection attempt identified and rejected. (model=mis

# System

In [4]:
df_system.columns

Index(['_id', '_index', '_score', '@timestamp', '@version',
       'event_code_version', 'event_hash', '@event_time_requested',
       'headers_content_type', 'headers_http_host',
       'headers_http_request_method', 'heathers_request_path',
       'sap_function_application', 'sap_source_type', 'sap_function_log_type',
       'sap_function_message', 'sap_app_env', 'region_id', 'region_name',
       'region_code', 'macro_region', 'service_id', 'http_status_code',
       'client_ip'],
      dtype='str')

In [5]:
lista_fila = df_system.iloc[0].tolist()
display(lista_fila)

['23bb795f-593c-4f5b-95f6-9ed2ee49d447',
 'sap-logs-2026.04',
 np.float64(0.7863),
 '2026-04-17T22:12:18.000Z',
 np.int64(1),
 '1.8.7',
 'bb746db61f0d38437ff6e0e4a3a425399f925aa0333352aecc14bb96853843b2',
 '2026-04-17T22:12:17.748Z',
 'application/json',
 'au-syd2.sap-services.internal',
 'GET',
 '/api/sap-fiori-launchpad/status',
 'sap-fiori-launchpad',
 'REST',
 'INFO',
 'Semantic cache hit — previous LLM response reused for sap-fiori-launchpad via REST.',
 'development',
 'AU-003',
 'Australia | Sydney (2)',
 'AU-SYD2',
 'Australia',
 'analytics-engine',
 np.float64(200.0),
 '208.223.2.195']

## Variables de las ventanas (system)

Cada ventana dura **5 minutos** y avanza cada **2.5 minutos**, por lo que cada log pertenece a **dos ventanas consecutivas** (traslape del 50%). Solo se conservan ventanas **completas**: se descartan la primera y la última del periodo, que tienen la mitad de los datos y distorsionan la detección.

### 1. Volumen y ritmo
- **total_requests**: número de logs en la ventana.
- **request_rate_std**: desviación estándar, en segundos, del tiempo entre solicitudes consecutivas dentro de la ventana. Se calcula con `@timestamp`, que tiene resolución de 1 segundo.
- **interval_cv**: coeficiente de variación (desviación estándar / promedio) del tiempo entre solicitudes, calculado con `@event_time_requested`, que tiene precisión de milisegundos. Valores altos indican tráfico en ráfagas; valores bajos, tráfico regular. Es independiente del volumen.

### 2. Origen del tráfico (IPs)
- **unique_ips_count**: cantidad de IPs distintas (`client_ip`).
- **ip_entropy**: entropía de Shannon normalizada de las solicitudes por IP. Vale 1 si todas las IPs generan lo mismo y baja hacia 0 si una IP concentra el tráfico.
- **ip_top_share**: fracción de las solicitudes generada por la IP más activa de la ventana. Complementa a `ip_entropy`.
- **novel_ips**: número de IPs vistas por primera vez en toda la base dentro de esta ventana. Las primeras 24 horas se toman como calentamiento y valen 0.

### 3. Respuestas HTTP
- **error_rate_400** y **error_rate_500**: proporción de códigos 4xx y 5xx sobre los códigos HTTP distintos de 200. El resto corresponde a otros códigos (p. ej. 3xx). Son composición entre los no-200, no tasa de fallos.
- **non_200_rate**: proporción de solicitudes con código distinto de 200 sobre el **total** de la ventana. Detecta aumentos de fallos que `error_rate_*` no muestra cuando la mezcla 4xx/5xx se mantiene.

### 4. Tipo de log (`sap_function_log_type`)
- **ratio_error**, **ratio_warn**, **ratio_security**, **ratio_info**, **ratio_audit**: proporción de logs de cada tipo sobre los logs con tipo asignado.
- `ratio_security` agrupa mensajes de seguridad de bajo riesgo, con redacción tranquilizadora (p. ej. "within limit", "no action taken"). `ratio_audit` incluye los eventos de auditoría más sensibles (intentos de escalamiento de privilegios, accesos no autorizados bloqueados, descargas masivas de datos), por lo que es el mejor indicador de actividad de seguridad entre los ratios.

### 5. Rutas y métodos
- **get_vs_post_ratio**: proporción de solicitudes GET sobre el total (el resto son POST u otros métodos).
- **novel_paths**: número de rutas (`heathers_request_path`) vistas por primera vez en la base dentro de esta ventana. Las primeras 24 horas valen 0.

### 6. Desviación de la mezcla de categorías (`js_*`)
Para cada columna categórica se calcula la **divergencia de Jensen-Shannon** (en bits, entre 0 y 1) entre la distribución de categorías de la ventana y la distribución global. 0 significa que la ventana es idéntica a lo habitual; sube cuando aparecen categorías nuevas o cambian las proporciones. Su nivel base depende del tamaño de la ventana y del número de categorías, por lo que se comparan ventanas entre sí y no columnas entre sí.
- **js_http_status_code**, **js_headers_http_request_method**, **js_sap_function_log_type**, **js_heathers_request_path**, **js_sap_function_application**, **js_sap_source_type**, **js_sap_app_env**, **js_region_code**, **js_headers_content_type**.
- `headers_http_host` se omite porque es equivalente 1:1 a `region_code`. `sap_function_message` no se incluye por ser texto libre.

### Variables descartadas
- **request_rate_avg**: el promedio de intervalos entre solicitudes es ≈ 300 / total_requests (correlación 0.9987), es decir, repite el volumen.
- **unique_paths_ratio**: hay ~67 rutas distintas por ventana, así que equivale a ≈ 67 / total_requests. Se reemplaza por `novel_paths` y `js_heathers_request_path`.

## Generar ventanas

In [6]:
window_size = pd.Timedelta(minutes=5)
window_step = pd.Timedelta(minutes=2.5)

# Se usa @timestamp como tiempo del evento para construir las ventanas.
required_columns = [
    '@timestamp', 'http_status_code', 'client_ip', 'sap_function_log_type',
    'heathers_request_path', 'headers_http_request_method',
    # nuevas
    '@event_time_requested', 'sap_function_application', 'sap_source_type',
    'sap_app_env', 'region_code', 'headers_http_host', 'headers_content_type',
]

logs = df_system[required_columns].copy()
logs['timestamp'] = pd.to_datetime(logs['@timestamp'], utc=True, errors='coerce')

if logs['timestamp'].isna().any():
    invalid_count = logs['timestamp'].isna().sum()
    raise ValueError(f'Hay {invalid_count} logs sin un @timestamp válido.')

logs['http_status_code'] = pd.to_numeric(logs['http_status_code'], errors='coerce')
logs['log_type'] = logs['sap_function_log_type'].astype('string').str.upper()
logs['method'] = logs['headers_http_request_method'].astype('string').str.upper()

events = expand_to_windows(logs)

status = events['http_status_code']
log_type = events['log_type']
events['is_non_200'] = status.notna() & status.ne(200)
events['is_4xx'] = status.between(400, 499).fillna(False)
events['is_5xx'] = status.between(500, 599).fillna(False)
events['has_log_type'] = log_type.notna()
events['is_error'] = log_type.eq('ERROR').fillna(False)
events['is_warning'] = log_type.eq('WARNING').fillna(False)
events['is_security'] = log_type.eq('SECURITY').fillna(False)
events['is_audit'] = log_type.eq('AUDIT').fillna(False)
events['is_info'] = log_type.eq('INFO').fillna(False)
events['is_get'] = events['method'].eq('GET').fillna(False)

grouped = events.groupby('window_start', sort=True)
df_batch_system = grouped.agg(
    total_requests=('timestamp', 'size'),
    unique_ips_count=('client_ip', 'nunique'),
    unique_paths_count=('heathers_request_path', 'nunique'),
    non_200_count=('is_non_200', 'sum'),
    count_4xx=('is_4xx', 'sum'),
    count_5xx=('is_5xx', 'sum'),
    log_type_count=('has_log_type', 'sum'),
    count_error=('is_error', 'sum'),
    count_warn=('is_warning', 'sum'),
    count_security=('is_security', 'sum'),
    count_info=('is_info', 'sum'),
    count_audit=('is_audit', 'sum'),
    count_get=('is_get', 'sum'),
)

# Los intervalos entre solicitudes consecutivas se calculan dentro de cada ventana, en segundos.
events_sorted = events.sort_values(['window_start', 'timestamp'], kind='mergesort')
intervals = events_sorted.groupby('window_start', sort=False)['timestamp'].diff().dt.total_seconds()
interval_stats = intervals.groupby(events_sorted['window_start']).agg(['mean', 'std'])
df_batch_system['request_rate_avg'] = interval_stats['mean']
df_batch_system['request_rate_std'] = interval_stats['std']

# Tasas como fracciones [0, 1]; el denominador HTTP incluye códigos distintos de 200.
non_200_denominator = df_batch_system['non_200_count'].replace(0, np.nan)
df_batch_system['error_rate_400'] = df_batch_system['count_4xx'] / non_200_denominator
df_batch_system['error_rate_500'] = df_batch_system['count_5xx'] / non_200_denominator
log_type_denominator = df_batch_system['log_type_count'].replace(0, np.nan)
df_batch_system['ratio_error'] = df_batch_system['count_error'] / log_type_denominator
df_batch_system['ratio_warn'] = df_batch_system['count_warn'] / log_type_denominator
df_batch_system['ratio_security'] = df_batch_system['count_security'] / log_type_denominator
df_batch_system['ratio_info'] = df_batch_system['count_info'] / log_type_denominator
df_batch_system['ratio_audit'] = df_batch_system['count_audit'] / log_type_denominator
df_batch_system['unique_paths_ratio'] = (
    df_batch_system['unique_paths_count'] / df_batch_system['total_requests']
)
df_batch_system['get_vs_post_ratio'] = (
    df_batch_system['count_get'] / df_batch_system['total_requests']
)

# Entropía de Shannon normalizada: 0 = una IP; 1 = solicitudes uniformes entre IPs.
ip_counts = (
    events.dropna(subset=['client_ip'])
    .groupby(['window_start', 'client_ip'], observed=True)
    .size()
)
ip_probabilities = ip_counts / ip_counts.groupby(level=0).transform('sum')
shannon_entropy = -(ip_probabilities * np.log2(ip_probabilities)).groupby(level=0).sum()
unique_ips_per_window = ip_counts.groupby(level=0).size()
normalized_entropy = shannon_entropy.div(np.log2(unique_ips_per_window)).where(
    unique_ips_per_window > 1, 0
)
df_batch_system['ip_entropy'] = normalized_entropy.reindex(df_batch_system.index).fillna(0)

# El índice identifica el inicio de ventana; se eliminan contadores intermedios.
df_batch_system = df_batch_system.drop(
    columns=[
        'unique_paths_count', 'non_200_count', 'count_4xx', 'count_5xx',
        'log_type_count', 'count_error', 'count_warn', 'count_security',
        'count_info', 'count_audit', 'count_get',
    ]
)
df_batch_system.index.name = 'window_start'
df_batch_system = df_batch_system[
    [
        'total_requests', 'request_rate_std', 'request_rate_avg',
        'unique_ips_count', 'error_rate_400', 'error_rate_500',
        'ratio_error', 'ratio_warn', 'ratio_security', 'ratio_info',
        'ratio_audit', 'ip_entropy', 'unique_paths_ratio', 'get_vs_post_ratio',
    ]
]

extra = build_extra_features(events)
df_batch_system = df_batch_system.join(extra, how='inner')
df_batch_system = df_batch_system.drop(columns=['request_rate_avg', 'unique_paths_ratio'])

display(df_batch_system.head())

,total_requests,request_rate_std,unique_ips_count,error_rate_400,error_rate_500,ratio_error,ratio_warn,ratio_security,ratio_info,ratio_audit,...,js_sap_source_type,js_sap_app_env,js_region_code,js_headers_http_host,js_headers_content_type,ip_top_share,novel_ips,novel_paths,non_200_rate,interval_cv
window_start,,,,,,,,,,,,,,,,,,,,,
2026-04-11 00:00:00+00:00,594,0.697776,105,0.332061,0.152672,0.124579,0.190236,0.023569,0.402357,0.072391,...,0.000855,0.000524,0.041795,0.041795,0.001136,0.020202,0,0,0.441077,1.185189
2026-04-11 00:02:30+00:00,586,0.649259,105,0.384314,0.133333,0.119454,0.203072,0.025597,0.383959,0.078498,...,0.002468,0.000085,0.038127,0.038127,0.000362,0.027304,0,0,0.435154,1.079468
2026-04-11 00:05:00+00:00,546,0.637322,104,0.375527,0.122363,0.128205,0.186813,0.02381,0.393773,0.086081,...,0.004418,0.000999,0.031510,0.031510,0.000479,0.025641,0,0,0.434066,1.003364
2026-04-11 00:07:30+00:00,566,0.661462,104,0.363636,0.150198,0.136042,0.187279,0.024735,0.393993,0.09364,...,0.004139,0.004622,0.042822,0.042822,0.000967,0.021201,0,0,0.446996,1.088070
2026-04-11 00:10:00+00:00,612,0.656220,105,0.348485,0.155303,0.117647,0.20098,0.029412,0.374183,0.088235,...,0.002017,0.004354,0.039299,0.039299,0.000596,0.021242,0,0,0.431373,1.156995


In [7]:
category_burst_report(logs)

,columna,categoria,logs,share_global,buckets_activos,buckets_total,share_en_top3_buckets,rareza_x_concentracion
15,http_status_code,504.0,8111,0.006999,1970,2017,0.004932,0.000115
9,http_status_code,404.0,7914,0.006829,1969,2017,0.004549,0.000108
144,region_code,CN-SHA,10676,0.009213,1997,2017,0.004590,0.000046
252,headers_http_host,cn-sha.sap-services.internal,10676,0.009213,1997,2017,0.004590,0.000046
168,region_code,IT-HPE,10727,0.009257,1999,2017,0.004102,0.000037
276,headers_http_host,it-hpe.sap-services.internal,10727,0.009257,1999,2017,0.004102,0.000037
441,client_ip,97.21.142.76,10904,0.009409,1999,2017,0.003944,0.000035
208,region_code,US-IA2,10610,0.009156,1999,2017,0.003864,0.000034
316,headers_http_host,us-ia2.sap-services.internal,10610,0.009156,1999,2017,0.003864,0.000034
127,region_code,AU-VIC,10709,0.009241,1999,2017,0.003829,0.000034


In [8]:
from ventanas_mejoras import check_unused_columns
res = check_unused_columns(df_system)
for k, v in res.items():
    print('##', k); print(v)

## buckets_con_pocos_logs
timestamp
2026-04-18 00:00:00+00:00    3
dtype: int64
## lag_describe
count    1158847.000
mean           0.250
std            0.144
min            0.001
1%             0.006
50%            0.250
99%            0.495
max            0.500
dtype: float64
## lag_max_z_bucket_mean
4.261668645592568
## score_describe
count    1.158847e+06
mean     7.498000e-01
std      1.445000e-01
min      5.000000e-01
25%      6.247000e-01
50%      7.498000e-01
75%      8.750000e-01
max      1.000000e+00
Name: _score, dtype: float64
## score_max_z_bucket_mean
3.1406557986019603
## score_max_z_bucket_std
3.4016021786068706
## score_mean_by_sap_function_log_type
sap_function_log_type
AUDIT       0.7502
DEBUG       0.7501
ERROR       0.7493
INFO        0.7500
PERF        0.7495
SECURITY    0.7501
WARNING     0.7498
Name: _score, dtype: float64
## score_mean_by_http_status_code
http_status_code
200.0    0.7499
201.0    0.7496
204.0    0.7503
206.0    0.7517
301.0    0.7495
302.0    0

In [9]:
from ventanas_mejoras import message_family_report
mf = message_family_report(df_system)
print(mf['n_familias'])
print(mf['frecuencia_familias'])
print(mf['familias_raras_x_log_type'])
print(mf['burst_familias'])

272
                                                    logs     share
msg_family                                                        
LLM context window at <N>% capacity for <APP> r...  7625   0.00658
Certificate pinning mismatch detected for <APP>...  7529  0.006497
LLM token cost for <APP> exceeded daily soft li...  7480  0.006455
Fallback configuration applied for <APP>.           7470  0.006446
Partial data returned from <APP>.                   7469  0.006445
...                                                  ...       ...
User accessed <APP> from an unregistered device...  1131  0.000976
Access pattern slightly different from recent h...  1125  0.000971
Payload marginally larger than average on <APP>...  1125  0.000971
Sequential user lookups on <APP> via <SRC> — po...  1120  0.000966
Request from a new IP range observed on <APP> —...  1101   0.00095

[272 rows x 2 columns]
sap_function_log_type                               AUDIT  DEBUG  ERROR  INFO  \
msg_family          

In [10]:
from ventanas_mejoras import poisson_burst_test, message_families
d = df_system.copy()
d['timestamp'] = pd.to_datetime(d['@timestamp'], utc=True)
d['msg_family'] = message_families(d)
for col in ['msg_family', 'sap_function_log_type', 'http_status_code', 'client_ip',
            'heathers_request_path', 'sap_function_application', 'sap_source_type',
            'region_code', 'headers_http_request_method', 'sap_app_env']:
    print(poisson_burst_test(d, col, top=5).to_string(), '\n')

      columna                                                                     categoria                    bucket  observado  esperado  neg_log10_p  umbral_bonferroni  esperado_sin_rafagas  significativo
0  msg_family       Thread contention detected — <N>% of threads blocked in <APP> on <SRC>. 2026-04-17 07:40:00+00:00         13      2.27         6.08               7.74                  5.74          False
1  msg_family      Same user logged in from two locations on <APP> — common with VPN usage. 2026-04-17 03:40:00+00:00          7      0.56         5.68               7.74                  5.74          False
2  msg_family  Cache hit ratio dropped to <N>% for <APP> — increased backend load on <SRC>. 2026-04-14 16:15:00+00:00         12      2.12         5.60               7.74                  5.74          False
3  msg_family                              Dependency <SRC> responded within SLA for <APP>. 2026-04-15 07:40:00+00:00         11      1.82         5.47               7.

## Convertir a CSV

In [12]:
df_batch_system['window_index'] = range(1, len(df_batch_system) + 1)
df_batch_system.to_csv('../data/ventanas_separadas/batch_system_v3.csv', index=True, float_format='%.6f')

# LLM

In [13]:
df_llm.columns

Index(['_id', '_index', '_score', '@timestamp', '@version',
       'event_code_version', 'event_hash', '@event_time_requested',
       'headers_content_type', 'headers_http_host',
       'headers_http_request_method', 'heathers_request_path',
       'sap_function_application', 'sap_source_type', 'sap_function_log_type',
       'sap_function_message', 'sap_app_env', 'sap_llm_response_size',
       'sap_llm_response_time', 'region_id', 'region_name', 'region_code',
       'macro_region', 'llm_model_id', 'llm_provider', 'llm_prompt_id',
       'llm_prompt_category', 'llm_prompt', 'llm_prompt_tokens',
       'llm_completion_tokens', 'llm_total_tokens', 'llm_response_time_ms',
       'llm_response_size_bytes', 'llm_status', 'llm_error_message',
       'llm_cost_usd', 'llm_temperature', 'llm_top_p', 'llm_stream',
       'llm_finish_reason'],
      dtype='str')

In [14]:
lista_fila = df_llm.iloc[0].tolist()
display(lista_fila)

['75225251-b3d5-4a76-97dc-9fcc5f29eeba',
 'llm-logs-2026.04',
 np.float64(0.953),
 '2026-04-12T12:57:39.000Z',
 np.int64(1),
 '1.8.9',
 'ad44a6f8dc6cbad0036218cbc65ae386ab53c846bfe6864b9053cef340af8ad7',
 '2026-04-12T12:57:38.340Z',
 'application/xml',
 'llm.nl-eem.sap-ai.internal',
 'POST',
 '/api/llm/sap-successfactors/completion',
 'sap-successfactors',
 'RFC',
 'LLM_TIMEOUT',
 'LLM TIMEOUT: Request timed out after 30 s. (model=gpt-5.4, app=sap-successfactors)',
 'qa',
 np.float64(nan),
 np.float64(31911.58),
 'EU-023',
 'Netherlands | Eemshaven',
 'NL-EEM',
 'Europe',
 'gpt-5.4',
 'OpenAI',
 'supply-chain-shortage-alert',
 'Supply Chain',
 'A critical component shortage has been flagged for sap-successfactors in Netherlands | Eemshaven. Draft an escalation plan and identify alternative suppliers.',
 np.float64(1451.0),
 np.float64(1229.0),
 np.float64(2680.0),
 np.float64(31911.58),
 np.float64(nan),
 'timeout',
 'Request timed out after 30 s.',
 np.float64(0.0536),
 np.float64(0.2

Posibles columnas para las ventanas de 5 minutos:
- total_llm_requests (Conteo total de logs)
- avg_llm_cost_usd (Promedio de los costos de las llamadas)
- total_llm_cost_usd (Suma de los costos de las llamadas)
- avg_token_ratio (llm_completion_tokens / llm_prompt_tokens. Para comparar el tamaño de los prompts de entrada con los de salida)
- avg_response_time_ms (Promedio de lo que tarda en responder el llm. Obtenido de llm_response_time_ms)

--

- llm_error_rate (Porcentaje de peticiones con llm_status = "error")
- llm_timeout_rate (Porcentaje de peticiones con llm_status = "timeout")

--

- temperature_variance (Qué tan dispersas están las temperaturas de las llamadas. Obtenido de llm_temperature)
- prompt_category_entropy (Entropía de Shannon normalizada entre 0 y 1; 1 indica categorías igualmente frecuentes. Obtenido de llm_prompt_category)
- stream_ratio (Porcentaje de llamadas en las que llm_stream = True)

--

- avg_prompt_length (Longitud promedio de los prompts. Obtenido de llm_prompt)
- max_prompt_length (Longitud del prompt más largo. Obtenido de llm_prompt)


Además de estas variables, estaría interesante agregar algo como "count_math_sus_words" donde tengamos una lista de palabras sospechosas y se contara cuántas veces aparecieron esas palabras en los prompts (como "bypass", "ignore al previous instructions", "give me the password"...)

In [16]:
window_size = pd.Timedelta(minutes=5)
window_step = pd.Timedelta(minutes=2.5)

required_columns = [
    '@timestamp',
    'llm_cost_usd',
    'llm_completion_tokens',
    'llm_prompt_tokens',
    'llm_response_time_ms',
    'llm_status',
    'llm_temperature',
    'llm_prompt_category',
    'llm_stream',
    'llm_prompt',
    'sap_function_message'   
]
logs = df_llm[required_columns].copy()
logs['timestamp'] = pd.to_datetime(logs['@timestamp'], utc=True, errors='coerce')

if logs['timestamp'].isna().any():
    invalid_count = logs['timestamp'].isna().sum()
    raise ValueError(f'Hay {invalid_count} logs sin un @timestamp válido.')

numeric_columns = [
    'llm_cost_usd',
    'llm_completion_tokens',
    'llm_prompt_tokens',
    'llm_response_time_ms',
    'llm_temperature',
]
for column in numeric_columns:
    logs[column] = pd.to_numeric(logs[column], errors='coerce')

logs['status'] = logs['llm_status'].astype('string').str.upper()
logs['token_ratio'] = logs['llm_completion_tokens'].div(
    logs['llm_prompt_tokens'].replace(0, np.nan)
)
logs['prompt_length'] = logs['llm_prompt'].astype('string').str.len()
logs['is_error'] = logs['status'].eq('ERROR').fillna(False)
logs['is_timeout'] = logs['status'].eq('TIMEOUT').fillna(False)
logs['is_stream'] = (
    logs['llm_stream'].astype('string').str.lower().eq('true').fillna(False)
)
logs['is_injection'] = logs['sap_function_message'].astype('string').str.contains(
    'injection|adversarial', case=False, regex=True, na=False)

events = expand_to_windows(logs)

grouped = events.groupby('window_start', sort=True)
df_batch_llm = grouped.agg(
    total_llm_requests=('timestamp', 'size'),
    avg_llm_cost_usd=('llm_cost_usd', 'mean'),
    ratio_injection=('is_injection', 'mean'),
    avg_token_ratio=('token_ratio', 'mean'),
    avg_response_time_ms=('llm_response_time_ms', 'mean'),
    llm_error_rate=('is_error', 'mean'),
    llm_timeout_rate=('is_timeout', 'mean'),
    temperature_variance=('llm_temperature', 'var'),
    stream_ratio=('is_stream', 'mean'),
    avg_prompt_length=('prompt_length', 'mean'),
    max_prompt_length=('prompt_length', 'max'),
)

# Normalized Shannon entropy is 0 for one category and 1 for a uniform distribution.
category_counts = (
    events.dropna(subset=['llm_prompt_category'])
    .groupby(['window_start', 'llm_prompt_category'], observed=True)
    .size()
)
category_probabilities = (
    category_counts / category_counts.groupby(level=0).transform('sum')
)
category_entropy = -(
    category_probabilities * np.log2(category_probabilities)
).groupby(level=0).sum()
categories_per_window = category_counts.groupby(level=0).size()
normalized_category_entropy = category_entropy.div(
    np.log2(categories_per_window)
).where(categories_per_window > 1, 0)
df_batch_llm['prompt_category_entropy'] = (
    normalized_category_entropy.reindex(df_batch_llm.index).fillna(0)
)

df_batch_llm.index.name = 'window_start'
df_batch_llm = df_batch_llm[
    [
        'total_llm_requests',
        'avg_llm_cost_usd',
        'ratio_injection',
        'avg_token_ratio',
        'avg_response_time_ms',
        'llm_error_rate',
        'llm_timeout_rate',
        'temperature_variance',
        'prompt_category_entropy',
        'stream_ratio',
        'avg_prompt_length',
        'max_prompt_length',
    ]
]

display(df_batch_llm.head())

,total_llm_requests,avg_llm_cost_usd,ratio_injection,avg_token_ratio,avg_response_time_ms,llm_error_rate,llm_timeout_rate,temperature_variance,prompt_category_entropy,stream_ratio,avg_prompt_length,max_prompt_length
window_start,,,,,,,,,,,,
2026-04-11 00:00:00+00:00,394,0.012840,0.0,1.313032,8720.057462,0.195431,0.093909,0.177682,0.959043,0.51269,110.560914,174
2026-04-11 00:02:30+00:00,417,0.013249,0.002398,1.262083,8502.081415,0.211031,0.083933,0.177055,0.974366,0.520384,111.179856,173
2026-04-11 00:05:00+00:00,396,0.011730,0.005051,1.335657,7923.143106,0.219697,0.070707,0.186211,0.985586,0.494949,111.194444,169
2026-04-11 00:07:30+00:00,391,0.012933,0.002558,1.424676,8657.928491,0.235294,0.117647,0.184881,0.976698,0.514066,110.296675,169
2026-04-11 00:10:00+00:00,381,0.013463,0.002625,1.379441,8867.140341,0.220472,0.112861,0.183340,0.967252,0.538058,110.83727,170


In [17]:
from ventanas_mejoras import poisson_burst_test
for col in ['llm_prompt_category', 'llm_model_id', 'llm_provider', 'llm_finish_reason',
            'llm_error_message', 'llm_status', 'llm_stream']:
    if col in df_llm.columns:
        logs[col] = df_llm[col].values
        print(poisson_burst_test(logs, col, top=5).to_string(), '\n')

               columna       categoria                    bucket  observado  esperado  neg_log10_p  umbral_bonferroni  esperado_sin_rafagas  significativo
0  llm_prompt_category         Support 2026-04-17 00:50:00+00:00         50     27.65         4.08               6.35                  4.35          False
1  llm_prompt_category       Analytics 2026-04-12 02:05:00+00:00         46     25.69         3.72               6.35                  4.35          False
2  llm_prompt_category  API Operations 2026-04-17 12:10:00+00:00         60     36.35         3.70               6.35                  4.35          False
3  llm_prompt_category    Productivity 2026-04-11 07:10:00+00:00        102     70.89         3.52               6.35                  4.35          False
4  llm_prompt_category    Supply Chain 2026-04-16 18:05:00+00:00         49     28.78         3.43               6.35                  4.35          False 

        columna          categoria                    bucket  observ

In [18]:
vc = df_llm['llm_prompt'].value_counts()
print(len(vc)); print(vc.tail(15))

178995
llm_prompt
Draft an escalation message for the unresolved HTTP_401_UNAUTHORIZED issue in sap-ariba affecting Israel | Tel Aviv users.                                                     1
Draft an escalation message for the unresolved RFC_NO_AUTHORITY issue in sap-s4hana-sales affecting United Arab Emirates | Dubai (2) users.                                    1
Predict customer churn risk for sap-data-sphere accounts in India | Pune based on usage patterns.                                                                              1
Draft constructive feedback for the sap-intelligent-rpa team in USA | Arizona based on Q1 performance.                                                                         1
Document lessons learned from the sap-ariba Q1 delivery in India | Pune and recommend process improvements for next cycle.                                                     1
Summarise exit interview themes for employees who left sap-ariba in Australia | Victoria during Q

In [19]:
from ventanas_mejoras import message_family_report
mf_llm = message_family_report(df_llm)
print(mf_llm['mensajes_no_nulos'], mf_llm.get('n_familias'), mf_llm.get('aviso'))
print(mf_llm['frecuencia_familias'].to_string())
print(mf_llm['burst_familias'].to_string())

1.0 241 None
                                                                                              logs     share
msg_family                                                                                                  
LLM TIMEOUT: Request timed out after <N> s. (model=<MODEL>, app=<APP>)                       77048   0.09973
LLM SUCCESS: gemini-<N>.<N>-pro responded in <N>.<N> ms (<N> tokens) via <APP>.              56141  0.072668
LLM SUCCESS: o<N>-mini responded in <N>.<N> ms (<N> tokens) via <APP>.                       37476  0.048509
LLM SUCCESS: gemini-<N>.<N>-flash responded in <N>.<N> ms (<N> tokens) via <APP>.            37201  0.048153
LLM SUCCESS: claude-<N>-<N>-sonnet responded in <N>.<N> ms (<N> tokens) via <APP>.           37200  0.048151
LLM SUCCESS: claude-opus-<N>.<N> responded in <N>.<N> ms (<N> tokens) via <APP>.             18944  0.024521
LLM SUCCESS: gemini-<N>.<N>-flash-<N>b responded in <N>.<N> ms (<N> tokens) via <APP>.       18827  0.024369
LLM SU

## Convertir a CSV

In [20]:
df_batch_llm['window_index'] = range(1, len(df_batch_llm) + 1)
df_batch_llm.to_csv('../data/ventanas_separadas/batch_llm_v3.csv', index=True, float_format='%.6f')